# Output Parsers Demo

> Converted from `05_output_parsers_demo.py` - part of **01 LangChain Foundations**.

## Setup

In [ ]:
# ============ IMPORTS AND SETUP ===========================================
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
)
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model

load_dotenv()

parser = StrOutputParser()

prompt = ChatPromptTemplate.from_template("wire a short poem about {topic}")

llm = init_chat_model(model="gpt-4o-mini", temperature=0)

chain = prompt | llm | parser

response = chain.invoke({"topic": "nature"})

print(type(response))


# JsonOutputParser example
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

prompt = ChatPromptTemplate.from_template(
    "Return a JSON object with 'name' and 'age' for: {description}"
)

chain = prompt | llm | parser

result = chain.invoke({"description": "A 25-year-old developer named Alex"})
print(result)  # {'name': 'Alex', 'age': 25}



<class 'langchain_core.messages.base.TextAccessor'>
{'name': 'Alex', 'age': 25}


### `Person`

In [2]:
# PydanticOutputParser example
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# ============ PERSON ======================================================
class Person(BaseModel):
    name: str = Field(description="The person's name")
    age: int = Field(description="The person's age")
    occupation: str = Field(description="The person's occupation")

In [3]:
parser = PydanticOutputParser(pydantic_object=Person)

In [4]:
prompt = ChatPromptTemplate.from_template(
    "Return a JSON object with 'name', 'age', and 'occupation' for: {description}"
).partial(format_instructions=parser.get_format_instructions())

In [5]:
chain = prompt | llm | parser

In [6]:
result = chain.invoke({"description": "A 30-year-old artist named Maria"})

In [7]:
print(result)  # Person(name='Maria', age=30, occupation='artist')

name='Maria' age=30 occupation='artist'


### `MovieReview`

In [8]:
# ============ MOVIEREVIEW =================================================
# Structured Output
class MovieReview(BaseModel):
    title: str = Field(description="The title of the movie")
    review: str = Field(description="A brief review of the movie")
    rating: int = Field(description="The rating of the movie out of 10")

In [9]:
# Bind the schema to the model
structured_model = llm.with_structured_output(MovieReview)

In [10]:
result = structured_model.invoke("Review: Inception is a mind-bending thriller. 9/10")

In [11]:
print(result)  # MovieReview(title='Inception', review='A mind-bending thriller.', rating=9)

title='Inception' review="Inception is a mind-bending thriller that masterfully blends action with a complex narrative about dreams within dreams. Christopher Nolan's direction is impeccable, and the visual effects are stunning, creating a surreal experience that keeps you on the edge of your seat. The performances, especially by Leonardo DiCaprio, are compelling, adding depth to the intricate plot. The film challenges viewers to think critically and engage with its themes of reality and perception. A must-watch for fans of cerebral cinema!" rating=9


## Summary

Defined in this notebook:

- `Person()`
- `MovieReview()`